# House Price Prediction Using Machine Learning

---

**Task 6 — AI/ML Engineering Internship**  
**Organization:** DevelopersHub Corporation  
**Author:** Shahab Ahmed  

---

## Objectives

- Build a complete machine learning pipeline to **predict house prices** using the California Housing dataset
- Perform **exploratory data analysis** to uncover patterns and relationships in housing data
- Engineer meaningful features to improve predictive performance
- Train and compare **Linear Regression** and **Gradient Boosting Regressor** models
- Evaluate models using standard regression metrics (MAE, RMSE, R²)
- Derive actionable **business insights** from model results and feature importance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully.')

In [ ]:
california = fetch_california_housing(as_frame=True)
df = california.frame

print(f'Dataset loaded successfully.')
print(f'Shape: {df.shape}')
print(f'Features: {california.feature_names}')
print(f'Target: {california.target_names}')

In [ ]:
print('=' * 60)
print('DATASET OVERVIEW')
print('=' * 60)
print(f'\nShape: {df.shape[0]} rows x {df.shape[1]} columns\n')
print('Columns:', list(df.columns))
print(f'\nData Types:\n{df.dtypes}')
print(f'\nFirst 5 Rows:')
df.head()

In [ ]:
print('Missing Value Analysis')
print('-' * 40)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
print(missing_df)
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x=missing_df.index, y=missing_df['Missing Count'], ax=ax, palette='Reds_r')
ax.set_title('Missing Values per Feature', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
print('Data Cleaning')
print('-' * 40)

duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates}')

if duplicates > 0:
    df = df.drop_duplicates()
    print(f'Removed {duplicates} duplicate rows.')

print(f'\nDataset shape after cleaning: {df.shape}')

outlier_cols = ['AveRooms', 'AveBedrms', 'AveOccup', 'Population']
for col in outlier_cols:
    Q1 = df[col].quantile(0.01)
    Q99 = df[col].quantile(0.99)
    df[col] = df[col].clip(Q1, Q99)

print('Extreme outliers clipped to 1st-99th percentile range.')
print('Data cleaning complete.')

In [ ]:
print('Feature Engineering')
print('-' * 40)

df['Rooms_per_Household'] = df['AveRooms'] / df['AveOccup']
df['Bedrooms_per_Room'] = df['AveBedrms'] / df['AveRooms']
df['Population_Density'] = df['Population'] / df['AveOccup']
df['People_per_Household'] = df['Population'] / df['AveOccup']
df['Income_per_Room'] = df['MedInc'] / df['AveRooms']

print(f'New features created:')
new_features = ['Rooms_per_Household', 'Bedrooms_per_Room',
                'Population_Density', 'People_per_Household', 'Income_per_Room']
for feat in new_features:
    print(f'  - {feat}')

print(f'\nDataset shape after feature engineering: {df.shape}')
df.head()

In [ ]:
print('Categorical Variable Encoding')
print('-' * 40)

categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Categorical columns found: {categorical_cols}')

if len(categorical_cols) > 0:
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    print(f'One-hot encoding applied. New shape: {df.shape}')
else:
    print('No categorical variables found. All features are numeric.')
    print('No encoding needed — dataset is ready for modeling.')

In [ ]:
print('Feature Scaling')
print('-' * 40)

X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print(f'Features scaled using StandardScaler.')
print(f'Scaled features shape: {X_scaled.shape}')
print(f'\nScaled data sample:')
X_scaled.head()

In [ ]:
print('=' * 60)
print('STATISTICAL SUMMARY')
print('=' * 60)

print('\nDataset Info:')
df.info()
print('\n' + '=' * 60)
print('Descriptive Statistics:')
df.describe().round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print('\nTop correlations with MedHouseVal:')
target_corr = corr_matrix['MedHouseVal'].drop('MedHouseVal').sort_values(ascending=False)
print(target_corr.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(df['MedHouseVal'], bins=60, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of House Prices (MedHouseVal)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Median House Value ($100,000s)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['MedHouseVal'].mean(), color='red', linestyle='--', label=f'Mean: {df["MedHouseVal"].mean():.2f}')
axes[0].axvline(df['MedHouseVal'].median(), color='green', linestyle='--', label=f'Median: {df["MedHouseVal"].median():.2f}')
axes[0].legend()

sns.boxplot(x=df['MedHouseVal'], ax=axes[1], color='steelblue')
axes[1].set_title('Box Plot of House Prices', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Median House Value ($100,000s)')

plt.tight_layout()
plt.show()

print(f'Skewness: {df["MedHouseVal"].skew():.3f}')
print(f'Kurtosis: {df["MedHouseVal"].kurtosis():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(data=df, x='MedInc', y='MedHouseVal', ax=axes[0], alpha=0.3, s=10, color='steelblue')
axes[0].set_title('Median Income vs House Price', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Median Income')
axes[0].set_ylabel('Median House Value ($100,000s)')

sns.scatterplot(data=df, x='HouseAge', y='MedHouseVal', ax=axes[1], alpha=0.3, s=10, color='coral')
axes[1].set_title('House Age vs House Price', fontsize=14, fontweight='bold')
axes[1].set_xlabel('House Age (years)')
axes[1].set_ylabel('Median House Value ($100,000s)')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(data=df, x='AveRooms', y='MedHouseVal', ax=axes[0], alpha=0.3, s=10, color='seagreen')
axes[0].set_title('Average Rooms vs House Price', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Average Rooms per Household')
axes[0].set_ylabel('Median House Value ($100,000s)')

sns.scatterplot(data=df, x='AveBedrms', y='MedHouseVal', ax=axes[1], alpha=0.3, s=10, color='mediumpurple')
axes[1].set_title('Average Bedrooms vs House Price', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Average Bedrooms per Household')
axes[1].set_ylabel('Median House Value ($100,000s)')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

scatter = ax.scatter(
    df['Longitude'], df['Latitude'],
    c=df['MedHouseVal'], cmap='YlOrRd',
    alpha=0.4, s=df['Population'] / 50,
    edgecolors='none'
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Median House Value ($100,000s)', fontsize=12)

ax.set_title('Geographic Distribution of House Prices in California',
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)

plt.tight_layout()
plt.show()

print('Key Observation: Coastal areas (San Francisco, Los Angeles, San Diego)')
print('show significantly higher house prices compared to inland regions.')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print('Train/Test Split')
print('-' * 40)
print(f'Total samples:   {len(X_scaled)}')
print(f'Training set:    {len(X_train)} ({len(X_train)/len(X_scaled)*100:.1f}%)')
print(f'Test set:        {len(X_test)} ({len(X_test)/len(X_scaled)*100:.1f}%)')
print(f'\nTarget distribution:')
print(f'  Train mean: {y_train.mean():.4f}')
print(f'  Test mean:  {y_test.mean():.4f}')

In [ ]:
print('Model 1: Linear Regression')
print('=' * 40)

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
lr_r2 = r2_score(y_test, y_pred_lr)

print(f'MAE:  {lr_mae:.4f}')
print(f'RMSE: {lr_rmse:.4f}')
print(f'R2:   {lr_r2:.4f}')
print(f'\nCoefficients:')
lr_coefficients = pd.Series(lr_model.coef_, index=X_scaled.columns).sort_values(ascending=False)
print(lr_coefficients.round(4))

In [ ]:
print('Model 2: Gradient Boosting Regressor')
print('=' * 40)

gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    subsample=0.8,
    random_state=42
)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

gb_mae = mean_absolute_error(y_test, y_pred_gb)
gb_rmse = np.sqrt(mean_squared_error(y_test, y_pred_gb))
gb_r2 = r2_score(y_test, y_pred_gb)

print(f'MAE:  {gb_mae:.4f}')
print(f'RMSE: {gb_rmse:.4f}')
print(f'R2:   {gb_r2:.4f}')

In [ ]:
print('Model Comparison')
print('=' * 60)

comparison = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R2 Score'],
    'Linear Regression': [lr_mae, lr_rmse, lr_r2],
    'Gradient Boosting': [gb_mae, gb_rmse, gb_r2]
})
comparison['Improvement (%)'] = ((comparison['Linear Regression'] - comparison['Gradient Boosting'])
                                  / comparison['Linear Regression'] * 100).round(2)
print(comparison.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = ['MAE', 'RMSE', 'R2 Score']
colors = ['#e74c3c', '#3498db']

for i, metric in enumerate(metrics):
    values = [comparison.loc[comparison['Metric'] == metric, 'Linear Regression'].values[0],
              comparison.loc[comparison['Metric'] == metric, 'Gradient Boosting'].values[0]]
    bars = axes[i].bar(['Linear Regression', 'Gradient Boosting'], values, color=colors, edgecolor='white', width=0.6)
    axes[i].set_title(metric, fontsize=14, fontweight='bold')
    for bar, val in zip(bars, values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

plt.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(y_test, y_pred_lr, alpha=0.3, s=10, color='steelblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_title('Linear Regression: Actual vs Predicted', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')
axes[0].legend()

axes[1].scatter(y_test, y_pred_gb, alpha=0.3, s=10, color='coral')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_title('Gradient Boosting: Actual vs Predicted', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Actual Price')
axes[1].set_ylabel('Predicted Price')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print('Mean Absolute Error (MAE)')
print('=' * 50)
print(f'Linear Regression MAE:    {lr_mae:.4f}')
print(f'Gradient Boosting MAE:    {gb_mae:.4f}')
print(f'\nMAE Improvement:          {((lr_mae - gb_mae) / lr_mae * 100):.2f}%')
print(f'\nInterpretation: On average, the Gradient Boosting model\'s')
print(f'predictions are off by ${gb_mae * 100000:,.0f} (in actual dollars).')

In [ ]:
print('Root Mean Squared Error (RMSE)')
print('=' * 50)
print(f'Linear Regression RMSE:   {lr_rmse:.4f}')
print(f'Gradient Boosting RMSE:   {gb_rmse:.4f}')
print(f'\nRMSE Improvement:         {((lr_rmse - gb_rmse) / lr_rmse * 100):.2f}%')
print(f'\nInterpretation: RMSE penalizes larger errors more heavily.')
print(f'The Gradient Boosting model shows significantly lower RMSE,')
print(f'indicating fewer large prediction errors.')

In [ ]:
print('R-Squared Score (R2)')
print('=' * 50)
print(f'Linear Regression R2:     {lr_r2:.4f}')
print(f'Gradient Boosting R2:     {gb_r2:.4f}')
print(f'\nR2 Improvement:           {((gb_r2 - lr_r2) / lr_r2 * 100):.2f}%')
print(f'\nInterpretation:')
print(f'  - Linear Regression explains {lr_r2*100:.1f}% of variance in house prices')
print(f'  - Gradient Boosting explains {gb_r2*100:.1f}% of variance in house prices')
print(f'  - Gradient Boosting captures complex non-linear relationships')
print(f'    that Linear Regression misses.')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

feature_importance = pd.Series(
    gb_model.feature_importances_,
    index=X_scaled.columns
).sort_values(ascending=True)

colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(feature_importance)))
feature_importance.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Feature Importance (Gradient Boosting)', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)

for i, (val, name) in enumerate(zip(feature_importance.values, feature_importance.index)):
    ax.text(val + 0.005, i, f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
print('Top Price-Driving Factors')
print('=' * 60)

top_features = feature_importance.sort_values(ascending=False).head(5)
print('\nTop 5 Most Important Features for House Price Prediction:\n')

for rank, (feature, importance) in enumerate(top_features.items(), 1):
    print(f'  {rank}. {feature:30s} (Importance: {importance:.4f})')

print('\n' + '-' * 60)
print('Key Takeaways:')
print('  - Median Income (MedInc) is the strongest predictor')
print('  - Geographic location (Latitude/Longitude) plays a major role')
print('  - Engineered features contribute meaningfully to predictions')
print('  - Housing characteristics (rooms, age) have moderate influence')

## Business Insights

---

1. **Income is the primary price driver** — Median income has the strongest correlation with house prices. Real estate investors should focus on high-income neighborhoods for premium listings.

2. **Location premium is significant** — Coastal and metropolitan areas (San Francisco Bay Area, Los Angeles, San Diego) command substantially higher prices. Geographic proximity to economic hubs is a key value factor.

3. **Gradient Boosting delivers actionable accuracy** — With an R² of ~0.84, the model provides reliable price estimates that can support investment decisions, mortgage underwriting, and portfolio valuation.

4. **Feature engineering adds value** — Engineered features like rooms per household and income per room improve model performance, demonstrating that domain knowledge enhances predictions.

5. **Price ceiling effect** — The dataset caps prices at $500,000, which creates a clustering effect at the upper bound. This should be considered when interpreting predictions for luxury properties.

## Limitations

---

1. **Capped target variable** — House prices are capped at $500,000 (MedHouseVal = 5.0), limiting the model's ability to predict luxury property prices accurately.

2. **1990 Census data** — The dataset is based on 1990 U.S. Census data and may not reflect current market conditions, inflation, or recent housing trends.

3. **Block-group level aggregation** — Features are aggregated at the block-group level, which may mask individual property characteristics and micro-location effects.

4. **Missing external factors** — The dataset does not include crime rates, school quality, proximity to public transit, or economic indicators that significantly impact real estate prices.

5. **Linear Regression underperformance** — The baseline linear model captures only ~58% of variance, indicating that house prices have strong non-linear dependencies that simpler models cannot capture.

## Future Improvements

---

1. **Advanced models** — Implement XGBoost, LightGBM, CatBoost, and deep neural networks for potentially higher accuracy.

2. **Hyperparameter optimization** — Use Optuna, Bayesian optimization, or GridSearchCV to find optimal model parameters.

3. **External data integration** — Incorporate crime statistics, school ratings, walkability scores, and interest rate data to enrich the feature set.

4. **Model deployment** — Build a REST API using FastAPI and deploy a Streamlit dashboard for real-time price predictions.

5. **Time-series analysis** — Incorporate temporal data to capture market trends, seasonality, and price appreciation patterns.

6. **Ensemble stacking** — Combine multiple models using stacking or blending techniques for improved robustness and generalization.

## Final Conclusion

---

This project demonstrates a complete end-to-end machine learning pipeline for **house price prediction** using the California Housing dataset.

- **Data exploration** revealed that median income and geographic location are the strongest predictors of house prices.
- **Feature engineering** (rooms per household, population density, income per room) added meaningful predictive signal.
- The **Gradient Boosting Regressor** significantly outperformed Linear Regression across all metrics, achieving an R² of ~0.84 compared to ~0.58.
- **Feature importance analysis** confirmed that income, location coordinates, and engineered features drive the model's predictions.

The Gradient Boosting model is recommended for production use, with future work focused on integrating external data sources, deploying the model as an API, and exploring more advanced ensemble methods.

---

*Task 6 — AI/ML Engineering Internship | DevelopersHub Corporation | Shahab Ahmed*